# E2 — Pipeline de Entrenamiento, Validación y Backtesting
## Proyecto Final | Trading Algorítmico | UAI Magíster Finanzas 2026
**Opción 2: Sistema de Trading Sistemático con Señales de Machine Learning**

| Parámetro | Valor |
|---|---|
| **Activo** | SPY (S&P 500 ETF) |
| **Horizonte de predicción** | 5 días hábiles |
| **Split train/test** | 80% / 20% (temporal estricto) |
| **Período test OOS** | Abril 2024 → Abril 2026 |
| **Autores** | Sebastián Moscoso · Joaquín Ocare |

---

### Estructura del notebook

| Sección | Contenido | Autor |
|---|---|---|
| §1 | Configuración e imports | — |
| §2 | Carga de datos y split temporal | Sebastián |
| §3 | Walk-forward CV con purging y embargo | Sebastián |
| §4 | Optimización de hiperparámetros (Optuna) | Sebastián |
| §5 | Métricas por fold y evaluación OOS | Sebastián |
| §6 | SHAP values y feature importance | Sebastián |
| §7 | Conversión de señal a estrategia | Joaquín |
| §8 | Backtesting y benchmarks | Joaquín |
| §9 | Análisis de overfitting (Sharpe IS vs OOS) | Joaquín |
| §10 | Prueba de permutación | Joaquín |
| §11 | Estabilidad de features por fold | Joaquín |
| §12 | Export de resultados | Joaquín |

**Input requerido:** `features.csv`, `data_meta.json`  
**Output generado:** `predictions.csv`, `resultados_ml.json`, 8 gráficos


## §1 · Configuración e imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import json
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.dates as mdates
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report,
    RocCurveDisplay
)
from xgboost import XGBClassifier
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import shap

# ── Reproducibilidad ──────────────────────────────────────────────────────────
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ── Estilo de gráficos ────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi':        120,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.grid':         True,
    'grid.alpha':        0.3,
    'font.family':       'sans-serif',
})
sns.set_palette('tab10')
pd.set_option('display.float_format', '{:.4f}'.format)

# ── Parámetros globales ───────────────────────────────────────────────────────
COSTO_BPS    = 0.0005   # 5 bps por trade
N_FOLDS      = 5
EMBARGO_DAYS = 5        # = horizonte de predicción
N_OPTUNA     = 30       # trials de optimización por modelo
N_PERM       = 100      # iteraciones prueba de permutación

print("✓ Configuración completa")
print(f"  Costos de transacción : {COSTO_BPS*10000:.0f} bps por trade")
print(f"  Folds walk-forward    : {N_FOLDS}")
print(f"  Embargo               : {EMBARGO_DAYS} días")
print(f"  Optuna trials         : {N_OPTUNA} por modelo")


## §2 · Carga de datos y split temporal

El split 80/20 es **estrictamente temporal**: los primeros 80% de observaciones (orden
cronológico) conforman el conjunto de entrenamiento, y los últimos 20% el test OOS.
El test set no se toca hasta la §5 (evaluación final) y la §8 (backtesting).


In [ ]:
# ── Carga (contrato acordado con E1) ─────────────────────────────────────────
df = pd.read_csv('features.csv', index_col='date', parse_dates=True)
with open('data_meta.json') as f:
    meta = json.load(f)
FEATURE_COLS = meta['feature_cols']

print(f"✓ features.csv   : {df.shape[0]:,} obs × {df.shape[1]} columnas")
print(f"  Período         : {df.index[0].date()} → {df.index[-1].date()}")
print(f"  Features        : {len(FEATURE_COLS)}")
print(f"  Balance target  : "
      f"Clase 0 = {(df['target']==0).mean():.1%} | "
      f"Clase 1 = {(df['target']==1).mean():.1%}")

# ── Split temporal estricto 80/20 ─────────────────────────────────────────────
split_idx  = int(len(df) * 0.80)
SPLIT_DATE = df.index[split_idx]

df_train = df.iloc[:split_idx].copy()
df_test  = df.iloc[split_idx:].copy()

X_train  = df_train[FEATURE_COLS]
y_train  = df_train['target']
X_test   = df_test[FEATURE_COLS]
y_test   = df_test['target']

print(f"\n✓ Split temporal:")
print(f"  Train (80%) : {len(df_train):,} obs "
      f"[{df_train.index[0].date()} → {df_train.index[-1].date()}]")
print(f"  Test  (20%) : {len(df_test):,} obs  "
      f"[{df_test.index[0].date()}  → {df_test.index[-1].date()}]")
print(f"\n⚠  TEST SET BLOQUEADO — se accede únicamente en §5 y §8")


## §3 · Walk-forward cross-validation con purging y embargo

**Expanding window:** el conjunto de train crece en cada fold, capturando más historia.  
**Embargo:** se eliminan las `EMBARGO_DAYS` observaciones inmediatamente posteriores al
corte de train, evitando contaminación por autocorrelación del target a 5 días (López de
Prado, 2018).  
**Normalización intra-fold:** el `StandardScaler` se ajusta **solo** sobre el subconjunto
de train de cada fold y se aplica (`.transform`) al de validación.


In [ ]:
def walk_forward_folds(X, n_folds=N_FOLDS, embargo=EMBARGO_DAYS):
    """
    Genera índices de walk-forward CV con expanding window y embargo.

    Parámetros
    ----------
    X       : DataFrame ordenado cronológicamente
    n_folds : número de folds (≥ 5 según rúbrica)
    embargo : días de buffer entre el último día de train y el primero de val

    Retorna
    -------
    list[tuple[list[int], list[int]]]  → [(train_idx, val_idx), ...]
    """
    n         = len(X)
    min_train = int(n * 0.60)          # train mínimo: 60% del total
    val_space = n - min_train
    fold_size = val_space // n_folds

    folds = []
    for i in range(n_folds):
        train_end = min_train + i * fold_size
        val_start = train_end + embargo
        val_end   = min(val_start + fold_size, n)
        if val_start >= val_end:
            continue
        folds.append((list(range(train_end)), list(range(val_start, val_end))))
    return folds


FOLDS = walk_forward_folds(X_train)

print(f"✓ Walk-forward CV: {len(FOLDS)} folds | embargo = {EMBARGO_DAYS} días")
print()
for i, (tr, val) in enumerate(FOLDS):
    d_tr  = X_train.index[tr]
    d_val = X_train.index[val]
    print(f"  Fold {i+1}: "
          f"Train {len(tr):4,d} obs ({d_tr[0].date()} → {d_tr[-1].date()})  "
          f"Val {len(val):3,d} obs ({d_val[0].date()} → {d_val[-1].date()})")


## §4 · Optimización de hiperparámetros con Optuna

Se utiliza el sampler TPE (Tree-structured Parzen Estimator) con semilla fija para
reproducibilidad. La métrica objetivo es el **AUC-ROC promedio en los folds de CV**.
El test set OOS no participa en ningún momento de esta etapa.

El desbalance de clases (62.2% clase 1) se corrige explícitamente:
- **XGBoost:** `scale_pos_weight = n(clase 0) / n(clase 1)`
- **Random Forest:** `class_weight='balanced'`


In [ ]:
def evaluar_modelo_cv(X, y, folds, build_model_fn):
    """Evalúa AUC-ROC promedio de un modelo en todos los folds CV."""
    aucs = []
    for tr_idx, val_idx in folds:
        X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
        X_vl, y_vl = X.iloc[val_idx], y.iloc[val_idx]
        scaler = StandardScaler()
        X_tr_s = scaler.fit_transform(X_tr)
        X_vl_s = scaler.transform(X_vl)
        model  = build_model_fn()
        model.fit(X_tr_s, y_tr)
        prob   = model.predict_proba(X_vl_s)[:, 1]
        aucs.append(roc_auc_score(y_vl, prob))
    return float(np.mean(aucs))


SCALE_POS = float((y_train == 0).sum() / (y_train == 1).sum())

# ── XGBoost ───────────────────────────────────────────────────────────────────
print("Optimizando XGBoost ...")
def obj_xgb(trial):
    p = dict(
        n_estimators     = trial.suggest_int('n_estimators', 100, 500),
        max_depth        = trial.suggest_int('max_depth', 3, 8),
        learning_rate    = trial.suggest_float('learning_rate', 0.01, 0.30, log=True),
        subsample        = trial.suggest_float('subsample', 0.6, 1.0),
        colsample_bytree = trial.suggest_float('colsample_bytree', 0.6, 1.0),
        reg_alpha        = trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        reg_lambda       = trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        scale_pos_weight = SCALE_POS,
        random_state     = RANDOM_STATE,
        eval_metric      = 'logloss',
        verbosity        = 0,
    )
    return evaluar_modelo_cv(X_train, y_train, FOLDS,
                             lambda: XGBClassifier(**p))

study_xgb = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
)
study_xgb.optimize(obj_xgb, n_trials=N_OPTUNA, show_progress_bar=False)

PARAMS_XGB = study_xgb.best_params | {
    'scale_pos_weight': SCALE_POS,
    'random_state': RANDOM_STATE,
    'eval_metric': 'logloss',
    'verbosity': 0,
}
print(f"  ✓ XGBoost  — AUC-ROC CV: {study_xgb.best_value:.4f}")


# ── Random Forest ─────────────────────────────────────────────────────────────
print("Optimizando Random Forest ...")
def obj_rf(trial):
    p = dict(
        n_estimators     = trial.suggest_int('n_estimators', 100, 500),
        max_depth        = trial.suggest_int('max_depth', 3, 15),
        min_samples_split= trial.suggest_int('min_samples_split', 2, 20),
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10),
        max_features     = trial.suggest_categorical('max_features', ['sqrt', 'log2', 0.5]),
        class_weight     = 'balanced',
        random_state     = RANDOM_STATE,
        n_jobs           = -1,
    )
    return evaluar_modelo_cv(X_train, y_train, FOLDS,
                             lambda: RandomForestClassifier(**p))

study_rf = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
)
study_rf.optimize(obj_rf, n_trials=N_OPTUNA, show_progress_bar=False)

PARAMS_RF = study_rf.best_params | {
    'class_weight': 'balanced',
    'random_state': RANDOM_STATE,
    'n_jobs': -1,
}
print(f"  ✓ Random Forest — AUC-ROC CV: {study_rf.best_value:.4f}")

# ── Selección del modelo ganador (basada en CV, NUNCA en test) ─────────────────
if study_xgb.best_value >= study_rf.best_value:
    MODELO_GANADOR = 'XGBoost'
    PARAMS_GANADOR = PARAMS_XGB
else:
    MODELO_GANADOR = 'RandomForest'
    PARAMS_GANADOR = PARAMS_RF

print(f"\n✓ Modelo ganador: {MODELO_GANADOR}  "
      f"(AUC-ROC CV = {max(study_xgb.best_value, study_rf.best_value):.4f})")


## §5 · Métricas por fold y evaluación out-of-sample

Primero se reportan las métricas de cada fold para ambos modelos, lo que permite
detectar inestabilidad o cambios de régimen. Luego se desbloquea el test set OOS
**una única vez** con el modelo entrenado sobre el 100% del train.


In [ ]:
def metricas_por_fold(X, y, folds, params, modelo_nombre):
    """Entrena y evalúa el modelo en cada fold; retorna DataFrame de métricas."""
    rows = []
    for i, (tr_idx, val_idx) in enumerate(folds):
        X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
        X_vl, y_vl = X.iloc[val_idx], y.iloc[val_idx]
        scaler = StandardScaler()
        X_tr_s = scaler.fit_transform(X_tr)
        X_vl_s = scaler.transform(X_vl)
        if modelo_nombre == 'XGBoost':
            model = XGBClassifier(**params)
        else:
            model = RandomForestClassifier(**params)
        model.fit(X_tr_s, y_tr)
        y_pred = model.predict(X_vl_s)
        y_prob = model.predict_proba(X_vl_s)[:, 1]
        rows.append({
            'fold':      i + 1,
            'n_train':   len(tr_idx),
            'n_val':     len(val_idx),
            'accuracy':  accuracy_score(y_vl, y_pred),
            'precision': precision_score(y_vl, y_pred, zero_division=0),
            'recall':    recall_score(y_vl, y_pred, zero_division=0),
            'f1':        f1_score(y_vl, y_pred, zero_division=0),
            'auc_roc':   roc_auc_score(y_vl, y_prob),
        })
    return pd.DataFrame(rows)


df_folds_xgb = metricas_por_fold(X_train, y_train, FOLDS, PARAMS_XGB, 'XGBoost')
df_folds_rf  = metricas_por_fold(X_train, y_train, FOLDS, PARAMS_RF,  'RandomForest')

print("Walk-forward CV — XGBoost")
display(df_folds_xgb.set_index('fold').round(4))
print(f"  Media → Acc: {df_folds_xgb['accuracy'].mean():.4f} | "
      f"F1: {df_folds_xgb['f1'].mean():.4f} | "
      f"AUC: {df_folds_xgb['auc_roc'].mean():.4f}")

print("\nWalk-forward CV — Random Forest")
display(df_folds_rf.set_index('fold').round(4))
print(f"  Media → Acc: {df_folds_rf['accuracy'].mean():.4f} | "
      f"F1: {df_folds_rf['f1'].mean():.4f} | "
      f"AUC: {df_folds_rf['auc_roc'].mean():.4f}")


In [ ]:
# Gráfico métricas por fold
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Walk-Forward CV — Métricas por fold', fontweight='bold')

metricas_plot = ['accuracy', 'f1', 'auc_roc']
colores_plot  = ['#4A90D9', '#2ECC71', '#E74C3C']

for ax, (df_f, nombre) in zip(axes, [(df_folds_xgb, 'XGBoost'),
                                      (df_folds_rf,  'Random Forest')]):
    for met, col in zip(metricas_plot, colores_plot):
        ax.plot(df_f['fold'], df_f[met], marker='o', label=met.upper(),
                color=col, lw=2, markersize=7)
    ax.axhline(0.5, color='gray', lw=1, ls='--', alpha=0.6, label='Baseline')
    ax.set_title(nombre, fontweight='bold')
    ax.set_xlabel('Fold'); ax.set_ylabel('Score')
    ax.set_ylim(0.3, 0.9); ax.set_xticks(df_f['fold'])
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('metricas_por_fold.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ metricas_por_fold.png")


In [ ]:
# ── Evaluación OOS — se toca el test set UNA SOLA VEZ ────────────────────────
print("Desbloqueando test set OOS ...")
print(f"  Período: {df_test.index[0].date()} → {df_test.index[-1].date()} "
      f"({len(df_test):,} observaciones)\n")

scaler_final = StandardScaler()
X_train_s    = scaler_final.fit_transform(X_train)
X_test_s     = scaler_final.transform(X_test)

# Entrenar modelo ganador sobre el 100% del train
if MODELO_GANADOR == 'XGBoost':
    model_final = XGBClassifier(**PARAMS_XGB)
    model_alt   = RandomForestClassifier(**PARAMS_RF)
    nombre_alt  = 'Random Forest'
else:
    model_final = RandomForestClassifier(**PARAMS_RF)
    model_alt   = XGBClassifier(**PARAMS_XGB)
    nombre_alt  = 'XGBoost'

model_final.fit(X_train_s, y_train)
model_alt.fit(X_train_s, y_train)

y_pred_oos  = model_final.predict(X_test_s)
y_prob_oos  = model_final.predict_proba(X_test_s)[:, 1]
y_prob_alt  = model_alt.predict_proba(X_test_s)[:, 1]
y_pred_alt  = model_alt.predict(X_test_s)

# Métricas
ACC  = accuracy_score(y_test, y_pred_oos)
PREC = precision_score(y_test, y_pred_oos, zero_division=0)
REC  = recall_score(y_test, y_pred_oos, zero_division=0)
F1   = f1_score(y_test, y_pred_oos, zero_division=0)
AUC  = roc_auc_score(y_test, y_prob_oos)
AUC_ALT = roc_auc_score(y_test, y_prob_alt)

# Baseline: siempre clase mayoritaria
baseline_pred = np.ones(len(y_test), dtype=int)
ACC_BASE = accuracy_score(y_test, baseline_pred)

print(f"Métricas OOS — {MODELO_GANADOR}")
print(f"  Accuracy  : {ACC:.4f}")
print(f"  Precision : {PREC:.4f}")
print(f"  Recall    : {REC:.4f}")
print(f"  F1        : {F1:.4f}")
print(f"  AUC-ROC   : {AUC:.4f}  ← métrica principal")
print(f"\nComparación:")
print(f"  {MODELO_GANADOR:18s}: AUC = {AUC:.4f}")
print(f"  {nombre_alt:18s}: AUC = {AUC_ALT:.4f}")
print(f"  {'Baseline':18s}: AUC = 0.5000")

print("\nReporte de clasificación completo:")
print(classification_report(y_test, y_pred_oos,
                             target_names=['Bajada (0)', 'Subida (1)']))


In [ ]:
# Curva ROC comparativa
fig, ax = plt.subplots(figsize=(8, 6))
RocCurveDisplay.from_predictions(
    y_test, y_prob_oos,
    name=f'{MODELO_GANADOR} (AUC = {AUC:.3f})', ax=ax, color='#4A90D9')
RocCurveDisplay.from_predictions(
    y_test, y_prob_alt,
    name=f'{nombre_alt} (AUC = {AUC_ALT:.3f})', ax=ax, color='#E67E22')
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Baseline (AUC = 0.500)')
ax.set_title('Curva ROC — Evaluación Out-of-Sample', fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('curva_roc.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ curva_roc.png")


## §6 · SHAP Values — Importancia de features

Se utiliza `TreeExplainer`, que calcula los valores de Shapley exactos para modelos
basados en árboles en tiempo polinomial. El análisis se realiza sobre el conjunto de
entrenamiento (500 observaciones muestreadas) para maximizar representatividad y
minimizar tiempo de cómputo.


In [ ]:
sample_idx = np.random.choice(len(X_train_s), size=min(500, len(X_train_s)),
                              replace=False)
X_shap = pd.DataFrame(X_train_s, columns=FEATURE_COLS).iloc[sample_idx]

explainer   = shap.TreeExplainer(model_final)
shap_values = explainer.shap_values(X_shap)

# Para RF: shap_values es lista → tomar clase positiva
shap_vals = shap_values[1] if isinstance(shap_values, list) else shap_values

shap_importance = (
    pd.DataFrame({'feature': FEATURE_COLS,
                  'importance': np.abs(shap_vals).mean(axis=0)})
    .sort_values('importance', ascending=False)
    .reset_index(drop=True)
)

TOP_FEATURES = shap_importance['feature'].head(10).tolist()

print("Top 10 features por SHAP (importancia media absoluta):")
display(shap_importance.head(10).assign(rank=range(1,11)).set_index('rank').round(4))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

plt.sca(axes[0])
shap.summary_plot(shap_vals, X_shap, feature_names=FEATURE_COLS,
                  show=False, max_display=15, plot_size=None)
axes[0].set_title('SHAP — Beeswarm (impacto por observación)', fontweight='bold')

top10 = shap_importance.head(10)
colors = plt.cm.Blues(np.linspace(0.4, 0.9, 10))[::-1]
axes[1].barh(range(10), top10['importance'].values[::-1],
             color=colors, edgecolor='white')
axes[1].set_yticks(range(10))
axes[1].set_yticklabels(top10['feature'].values[::-1])
axes[1].set_title('SHAP — Importancia media absoluta (Top 10)', fontweight='bold')
axes[1].set_xlabel('Mean |SHAP value|')

plt.tight_layout()
plt.savefig('shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ shap_importance.png")


## §7 · Conversión de señal a estrategia de trading

Se implementan dos esquemas de sizing que permiten comparar la señal continua del
modelo con una versión binaria más simple:

- **Continua:** `pos(t) = 2 · P(retorno > 0 | X_t) − 1 ∈ [−1, 1]`  
  La exposición es proporcional a la confianza del modelo. Es la estrategia principal
  porque balancea señal y control de rotación.
  
- **Binaria:** `pos(t) = 1` si `P > 0.5`, `pos(t) = 0` en caso contrario.  
  Mayor rotación, mayor retorno bruto potencial, pero también mayor costo acumulado.

**Costos:** 5 bps por cambio de posición, calculados sobre el delta absoluto.


In [ ]:
# Construir DataFrame de predicciones OOS
pred = df_test[['fwd_return', 'target']].copy()
pred['prob_pred']  = y_prob_oos
pred['pred_label'] = y_pred_oos

# Posiciones
pred['pos_continua'] = 2 * pred['prob_pred'] - 1
pred['pos_binaria']  = np.where(pred['prob_pred'] > 0.5, 1.0, 0.0)

# Costos de transacción
delta_cont = pred['pos_continua'].diff().abs().fillna(0)
delta_bin  = pred['pos_binaria'].diff().abs().fillna(0)

# Retornos brutos y netos (shift(1): la posición se toma el día anterior)
pred['ret_continua'] = pred['pos_continua'].shift(1) * pred['fwd_return'] - delta_cont * COSTO_BPS
pred['ret_binaria']  = pred['pos_binaria'].shift(1)  * pred['fwd_return'] - delta_bin  * COSTO_BPS

pred_bt = pred.iloc[1:].copy()   # eliminar primera fila (NaN por shift)

print(f"✓ Posiciones calculadas sobre {len(pred_bt):,} observaciones OOS")
print(f"  Continua — rango : [{pred_bt['pos_continua'].min():.3f}, "
      f"{pred_bt['pos_continua'].max():.3f}]")
print(f"  Binaria  — long  : {(pred_bt['pos_binaria']==1).sum()} días | "
      f"flat: {(pred_bt['pos_binaria']==0).sum()} días")
print(f"  Costo total continua : {delta_cont.sum()*100:.3f}%")
print(f"  Costo total binaria  : {delta_bin.sum()*100:.3f}%")

# Guardar predictions.csv
pred_export = pred[['prob_pred', 'pred_label', 'target', 'fwd_return']].copy()
pred_export.index.name = 'date'
pred_export.to_csv('predictions.csv')
print("\n✓ predictions.csv guardado")


## §8 · Backtesting completo con benchmarks

Se comparan cuatro estrategias en el período OOS:

| Estrategia | Descripción |
|---|---|
| **XGB Continua** | Sizing proporcional a prob. predicha (estrategia principal) |
| **XGB Binaria** | Long/flat según umbral 0.5 |
| **Buy & Hold** | Posición 100% long permanente |
| **Momentum 20D** | Long si retorno acumulado 20D > 0, flat en caso contrario |


In [ ]:
def calcular_metricas(retornos: pd.Series, nombre: str = '') -> dict:
    """
    Métricas de backtest desde una serie de retornos periódicos.

    Retorna sharpe, calmar, max_drawdown, hit_rate, retorno_total.
    """
    r       = retornos.dropna()
    sharpe  = float((r.mean() / r.std()) * np.sqrt(252)) if r.std() > 1e-10 else 0.0
    equity  = (1 + r).cumprod()
    roll_mx = equity.cummax()
    dd      = (equity - roll_mx) / roll_mx
    max_dd  = float(dd.min())
    n_years = len(r) / 252
    ret_ann = float(equity.iloc[-1] ** (1 / n_years) - 1) if n_years > 0 else 0.0
    calmar  = ret_ann / abs(max_dd) if abs(max_dd) > 1e-10 else 0.0
    r_act   = r[r != 0]
    hit     = float((r_act > 0).mean() * 100) if len(r_act) else 0.0
    ret_tot = float((equity.iloc[-1] - 1) * 100)
    return {
        'sharpe':        round(sharpe, 4),
        'calmar':        round(calmar, 4),
        'max_drawdown':  round(max_dd, 4),
        'hit_rate':      round(hit, 2),
        'retorno_total': round(ret_tot, 2),
    }


# Benchmarks
ret_buyhold = pred_bt['fwd_return'].copy()

mom_signal  = pred['fwd_return'].rolling(20).sum().shift(1)
pos_mom_arr = np.where(mom_signal.reindex(pred_bt.index) > 0, 1.0, 0.0)
pos_mom     = pd.Series(pos_mom_arr, index=pred_bt.index)
delta_mom   = pos_mom.diff().abs().fillna(0)
ret_mom     = pos_mom.shift(1).fillna(0) * pred_bt['fwd_return'] - delta_mom * COSTO_BPS

# Métricas
MET_CONT    = calcular_metricas(pred_bt['ret_continua'], 'XGB Continua')
MET_BIN     = calcular_metricas(pred_bt['ret_binaria'],  'XGB Binaria')
MET_BH      = calcular_metricas(ret_buyhold,             'Buy & Hold')
MET_MOM     = calcular_metricas(ret_mom,                 'Momentum 20D')

tabla_bt = pd.DataFrame({
    'XGB Continua': MET_CONT,
    'XGB Binaria':  MET_BIN,
    'Buy & Hold':   MET_BH,
    'Momentum 20D': MET_MOM,
}).T

print("Métricas de backtesting — Test Set OOS")
display(tabla_bt.style
    .format('{:.4f}')
    .highlight_max(subset=['sharpe', 'calmar', 'hit_rate', 'retorno_total'],
                   color='#d5f0d5')
    .highlight_min(subset=['max_drawdown'], color='#f0d5d5'))


In [ ]:
# Equity curves + drawdown
eq_cont = (1 + pred_bt['ret_continua']).cumprod()
eq_bin  = (1 + pred_bt['ret_binaria']).cumprod()
eq_bh   = (1 + ret_buyhold).cumprod()
eq_mom  = (1 + ret_mom).cumprod()

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

ax1 = axes[0]
ax1.plot(eq_cont.index, eq_cont, label=f'XGB Continua (SR={MET_CONT["sharpe"]:.2f})',
         color='royalblue', lw=2)
ax1.plot(eq_bin.index,  eq_bin,  label=f'XGB Binaria  (SR={MET_BIN["sharpe"]:.2f})',
         color='steelblue', lw=2, ls='--')
ax1.plot(eq_bh.index,   eq_bh,   label=f'Buy & Hold   (SR={MET_BH["sharpe"]:.2f})',
         color='darkorange', lw=2)
ax1.plot(eq_mom.index,  eq_mom,  label=f'Momentum 20D (SR={MET_MOM["sharpe"]:.2f})',
         color='gray', lw=1.5, ls=':')
ax1.axhline(1, color='black', lw=0.8, alpha=0.4)
ax1.set_title('Equity Curve — Test Set OOS (Abril 2024 → Abril 2026)',
              fontweight='bold')
ax1.set_ylabel('Riqueza acumulada (base = 1)')
ax1.legend(loc='upper left')
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

ax2 = axes[1]
dd_cont = ((eq_cont - eq_cont.cummax()) / eq_cont.cummax()) * 100
dd_bh   = ((eq_bh   - eq_bh.cummax())   / eq_bh.cummax())   * 100
ax2.fill_between(dd_cont.index, dd_cont, 0, alpha=0.4, color='royalblue',
                 label='XGB Continua')
ax2.fill_between(dd_bh.index,   dd_bh,  0, alpha=0.3, color='darkorange',
                 label='Buy & Hold')
ax2.set_title('Drawdown (%)'); ax2.set_ylabel('Drawdown (%)')
ax2.legend()
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

plt.tight_layout()
plt.savefig('equity_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ equity_curve.png")


In [ ]:
# Heatmap de retornos mensuales
ret_mensual = pred_bt['ret_continua'].resample('ME').apply(
    lambda x: (1 + x).prod() - 1) * 100

ret_pivot = pd.DataFrame({
    'año': ret_mensual.index.year,
    'mes': ret_mensual.index.month,
    'ret': ret_mensual.values,
}).pivot(index='año', columns='mes', values='ret')

meses = ['Ene','Feb','Mar','Abr','May','Jun',
         'Jul','Ago','Sep','Oct','Nov','Dic']
ret_pivot.columns = [meses[m-1] for m in ret_pivot.columns]

fig, ax = plt.subplots(figsize=(13, 3))
sns.heatmap(ret_pivot, annot=True, fmt='.1f', cmap='RdYlGn', center=0,
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Retorno (%)'})
ax.set_title('Retornos Mensuales — Modelo XGBoost (posición continua, %)',
             fontweight='bold')
plt.tight_layout()
plt.savefig('retornos_mensuales.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ retornos_mensuales.png")


## §9 · Análisis de overfitting — Sharpe in-sample vs out-of-sample

El Sharpe in-sample se calcula aplicando el modelo sobre el propio conjunto de
entrenamiento (sin reentrenamiento). La brecha respecto al Sharpe OOS cuantifica
el grado de sobreajuste: a mayor brecha, mayor memorización de ruido histórico.

Un ratio OOS/IS > 0.20 se considera aceptable en la literatura de ML aplicado a
finanzas para activos de alta eficiencia como el SPY.


In [ ]:
# Sharpe in-sample
y_prob_is  = model_final.predict_proba(X_train_s)[:, 1]
pos_is     = 2 * y_prob_is - 1
ret_is     = pd.Series(pos_is, index=X_train.index).shift(1).fillna(0)              * df_train['fwd_return']
SHARPE_IS  = float((ret_is.mean() / ret_is.std()) * np.sqrt(252))              if ret_is.std() > 1e-10 else 0.0
SHARPE_OOS = MET_CONT['sharpe']
RATIO_IS_OOS = SHARPE_OOS / SHARPE_IS if SHARPE_IS != 0 else 0.0

print("Análisis de overfitting — Sharpe IS vs OOS")
print(f"  Sharpe in-sample  (entrenamiento): {SHARPE_IS:.4f}")
print(f"  Sharpe OOS        (test real)    : {SHARPE_OOS:.4f}")
print(f"  Ratio OOS / IS                   : {RATIO_IS_OOS:.4f}")
print()
if RATIO_IS_OOS > 0.20:
    print("  → Ratio > 0.20: nivel de generalización aceptable para activos líquidos.")
else:
    print("  → Ratio < 0.20: degradación severa; señal principalmente in-sample.")
print(f"\n  Interpretación: el modelo retiene ~{RATIO_IS_OOS:.0%} de su "
      f"desempeño IS al evaluarse OOS.")

# Gráfico
fig, ax = plt.subplots(figsize=(7, 5))
colores = ['#e74c3c', '#2ecc71']
barras  = ax.bar(['In-Sample', 'Out-of-Sample'],
                 [SHARPE_IS, SHARPE_OOS],
                 color=colores, width=0.45, edgecolor='white')
ax.axhline(1.0, color='navy', lw=1.5, ls='--', label='SR = 1.0 (referencia)')
for bar, val in zip(barras, [SHARPE_IS, SHARPE_OOS]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{val:.2f}', ha='center', fontweight='bold', fontsize=13)
ax.set_title('Sharpe Ratio: In-Sample vs Out-of-Sample\n'
             '(Evidencia de Overfitting)', fontweight='bold')
ax.set_ylabel('Sharpe Ratio Anualizado')
ax.legend()
plt.tight_layout()
plt.savefig('overfitting_sharpe.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ overfitting_sharpe.png")


## §10 · Prueba de permutación

**H₀:** La señal del modelo no difiere estadísticamente del azar.

El procedimiento barre aleatoriamente las probabilidades predichas, rompiendo toda
relación entre features y retornos. Se repite `N_PERM` veces para construir la
distribución nula del Sharpe bajo H₀. Un p-value < 0.05 rechaza H₀ al 5%.


In [ ]:
np.random.seed(RANDOM_STATE)
probs_oos    = pred_bt['prob_pred'].values.copy()
retornos_oos = pred_bt['fwd_return'].values.copy()
sharpe_real  = SHARPE_OOS

sharpes_perm = []
for _ in range(N_PERM):
    probs_p = np.random.permutation(probs_oos)
    pos_p   = 2 * probs_p - 1
    ret_p   = np.roll(pos_p, 1) * retornos_oos
    ret_p   = ret_p[1:]
    sr      = float((ret_p.mean() / ret_p.std()) * np.sqrt(252))               if ret_p.std() > 1e-10 else 0.0
    sharpes_perm.append(sr)

sharpes_perm = np.array(sharpes_perm)
P_VALUE      = float((sharpes_perm >= sharpe_real).sum() / N_PERM)
PCTIL_95     = float(np.percentile(sharpes_perm, 95))

print(f"Prueba de permutación ({N_PERM} iteraciones)")
print(f"  Sharpe real       : {sharpe_real:.4f}")
print(f"  Sharpe perm media : {sharpes_perm.mean():.4f} ± {sharpes_perm.std():.4f}")
print(f"  Percentil 95      : {PCTIL_95:.4f}")
print(f"  p-value           : {P_VALUE:.4f}")
print()
if P_VALUE < 0.05:
    print(f"  → p = {P_VALUE:.3f} < 0.05 → RECHAZAMOS H₀ — señal estadísticamente real.")
elif P_VALUE < 0.10:
    print(f"  → p = {P_VALUE:.3f} < 0.10 → Evidencia débil de señal.")
else:
    print(f"  → p = {P_VALUE:.3f} ≥ 0.05 → No se puede rechazar H₀.")

# Gráfico
fig, ax = plt.subplots(figsize=(11, 5))
ax.hist(sharpes_perm, bins=25, color='#95a5a6', edgecolor='white',
        alpha=0.8, label='Sharpe permutado (H₀)')
ax.axvline(sharpe_real, color='royalblue', lw=2.5, ls='--',
           label=f'Sharpe real = {sharpe_real:.3f}')
ax.axvline(PCTIL_95, color='red', lw=1.5, ls=':',
           label=f'Percentil 95 = {PCTIL_95:.3f}')
ax.set_xlabel('Sharpe Ratio Anualizado')
ax.set_ylabel('Frecuencia')
ax.set_title(f'Prueba de Permutación ({N_PERM} iteraciones) | p-value = {P_VALUE:.3f}',
             fontweight='bold')
ax.text(0.97, 0.92, f'p-value = {P_VALUE:.3f}',
        transform=ax.transAxes, ha='right', va='top', fontsize=12,
        color='green' if P_VALUE < 0.05 else 'red',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))
ax.legend()
plt.tight_layout()
plt.savefig('permutation_test.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ permutation_test.png")


## §11 · Estabilidad de métricas por fold y análisis de features

Se comparan las métricas de XGBoost y Random Forest en cada fold para detectar
inestabilidad o cambios de régimen que afecten diferencialmente a cada modelo.


In [ ]:
folds_xgb_data = df_folds_xgb.to_dict(orient='records')
folds_rf_data  = df_folds_rf.to_dict(orient='records')

metricas_plot = ['auc_roc', 'f1', 'accuracy', 'precision', 'recall']
fig, axes = plt.subplots(1, len(metricas_plot), figsize=(17, 4), sharey=False)

for ax, met in zip(axes, metricas_plot):
    xgb_vals = [f[met] for f in folds_xgb_data]
    rf_vals  = [f[met] for f in folds_rf_data]
    folds_n  = [f['fold'] for f in folds_xgb_data]
    ax.plot(folds_n, xgb_vals, 'o-', color='royalblue',
            label='XGBoost', lw=2, markersize=6)
    ax.plot(folds_n, rf_vals,  's--', color='tomato',
            label='RF', lw=2, markersize=6)
    ax.axhline(0.5, color='gray', lw=0.8, ls=':', alpha=0.6)
    ax.set_title(met.upper().replace('_', '-'), fontweight='bold')
    ax.set_xlabel('Fold'); ax.set_xticks(folds_n)
    if ax == axes[0]:
        ax.legend(fontsize=9)

fig.suptitle('Estabilidad de Métricas por Fold — XGBoost vs Random Forest',
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('estabilidad_folds.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ estabilidad_folds.png")


## §12 · Export de resultados y dashboard final

In [ ]:
# ── Construir resultados_ml.json ──────────────────────────────────────────────
def to_py(v):
    """Convierte tipos numpy a Python nativo para serialización JSON."""
    if isinstance(v, (np.floating, np.integer)):
        return float(v)
    if isinstance(v, np.ndarray):
        return v.tolist()
    return v

resultados_ml = {
    'modelo_ganador': MODELO_GANADOR,
    'metricas_oos': {
        'accuracy':  round(ACC, 4),
        'precision': round(PREC, 4),
        'recall':    round(REC, 4),
        'f1':        round(F1, 4),
        'auc_roc':   round(AUC, 4),
    },
    'backtest_modelo': {k: to_py(v) for k, v in MET_CONT.items()},
    'backtest_modelo_binario': {k: to_py(v) for k, v in MET_BIN.items()},
    'backtest_buyhold':  {k: to_py(v) for k, v in MET_BH.items()},
    'backtest_momentum': {k: to_py(v) for k, v in MET_MOM.items()},
    'sharpe_in_sample':  round(SHARPE_IS, 4),
    'sharpe_out_sample': round(SHARPE_OOS, 4),
    'overfitting_ratio': round(RATIO_IS_OOS, 4),
    'permutation_pvalue':       round(P_VALUE, 4),
    'permutation_sharpe_media': round(float(sharpes_perm.mean()), 4),
    'permutation_sharpe_std':   round(float(sharpes_perm.std()), 4),
    'permutation_n_iteraciones': N_PERM,
    'top_features_shap': TOP_FEATURES,
    'metricas_por_fold': {
        'xgboost':       folds_xgb_data,
        'random_forest': folds_rf_data,
    },
    'meta_datos': {
        'ticker':         meta['ticker'],
        'start_date':     meta['start_date'],
        'end_date':       meta['end_date'],
        'n_obs_total':    meta['n_obs'],
        'n_obs_oos':      len(pred_bt),
        'horizonte_dias': meta['horizonte_dias'],
        'split_date':     str(SPLIT_DATE.date()),
        'costo_bps':      COSTO_BPS * 10000,
    },
}

with open('resultados_ml.json', 'w', encoding='utf-8') as f:
    json.dump(resultados_ml, f, indent=2, ensure_ascii=False)

print("✓ resultados_ml.json guardado")
print()
print("===== RESUMEN FINAL =====")
for k, v in [
    ('Modelo',              MODELO_GANADOR),
    ('AUC-ROC OOS',        f"{AUC:.4f}"),
    ('Sharpe IS',          f"{SHARPE_IS:.4f}"),
    ('Sharpe OOS',         f"{SHARPE_OOS:.4f}"),
    ('Ratio OOS/IS',       f"{RATIO_IS_OOS:.4f}"),
    ('Max Drawdown OOS',   f"{MET_CONT['max_drawdown']:.4f}"),
    ('Retorno Total OOS',  f"{MET_CONT['retorno_total']:.2f}%"),
    ('p-value permutación',f"{P_VALUE:.4f}"),
]:
    print(f"  {k:<25}: {v}")


In [ ]:
# Dashboard final de resumen
fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# Equity curve (ancho completo)
ax1 = fig.add_subplot(gs[0, :])
ax1.plot(eq_cont.index, eq_cont, label=f'XGB Continua (SR={MET_CONT["sharpe"]:.2f})',
         color='royalblue', lw=2)
ax1.plot(eq_bh.index,  eq_bh,   label=f'Buy & Hold   (SR={MET_BH["sharpe"]:.2f})',
         color='darkorange', lw=2)
ax1.plot(eq_mom.index, eq_mom,  label=f'Momentum 20D (SR={MET_MOM["sharpe"]:.2f})',
         color='gray', lw=1.5, ls='--')
ax1.axhline(1, color='black', lw=0.8, alpha=0.4)
ax1.set_title('Equity Curve — Test Set OOS', fontweight='bold')
ax1.legend(); ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

# Sharpe IS vs OOS
ax2 = fig.add_subplot(gs[1, 0])
ax2.bar(['IS', 'OOS'], [SHARPE_IS, SHARPE_OOS],
        color=['#e74c3c', '#27ae60'], width=0.4, edgecolor='white')
ax2.axhline(1, color='navy', lw=1.5, ls='--', alpha=0.7)
ax2.set_title('Sharpe IS vs OOS\n(Overfitting)', fontweight='bold')
ax2.set_ylabel('Sharpe Anualizado')
for i, v in enumerate([SHARPE_IS, SHARPE_OOS]):
    ax2.text(i, v + 0.2, f'{v:.2f}', ha='center', fontweight='bold')

# Distribución permutación
ax3 = fig.add_subplot(gs[1, 1])
ax3.hist(sharpes_perm, bins=20, color='#95a5a6', edgecolor='white', alpha=0.8)
ax3.axvline(sharpe_real, color='royalblue', lw=2.5, ls='--',
            label=f'SR real = {sharpe_real:.2f}')
ax3.set_title(f'Prueba Permutación\np-value = {P_VALUE:.3f}', fontweight='bold')
ax3.set_xlabel('Sharpe Permutado'); ax3.legend(fontsize=9)

# Top 5 features SHAP
ax4 = fig.add_subplot(gs[1, 2])
top5   = shap_importance['feature'].head(5).tolist()
imp5   = shap_importance['importance'].head(5).values
colors = plt.cm.Blues(np.linspace(0.5, 0.9, 5))
ax4.barh(range(5), imp5[::-1], color=colors, edgecolor='white')
ax4.set_yticks(range(5)); ax4.set_yticklabels(top5[::-1])
ax4.set_title('Top 5 Features SHAP', fontweight='bold')
ax4.set_xlabel('Mean |SHAP value|')

fig.suptitle(
    f'Dashboard Final — Proyecto Trading Algorítmico UAI\n'
    f'SPY | {MODELO_GANADOR} | Test OOS: '
    f'{pred_bt.index[0].date()} → {pred_bt.index[-1].date()}',
    fontsize=14, fontweight='bold', y=1.02
)
plt.savefig('dashboard_final.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ dashboard_final.png")
print()
print("Archivos generados:")
for f in ['predictions.csv','resultados_ml.json','metricas_por_fold.png',
          'curva_roc.png','shap_importance.png','equity_curve.png',
          'overfitting_sharpe.png','permutation_test.png',
          'estabilidad_folds.png','retornos_mensuales.png','dashboard_final.png']:
    print(f"  ✓ {f}")
